In [3]:
import tkinter as tk
from tkinter import messagebox
import random
import math

try:
    import winsound
    HAS_WINSOUND = True
except ImportError:
    HAS_WINSOUND = False


# ---------------------------------------------------------------------------
# Sound
# ---------------------------------------------------------------------------
class SoundManager:
    """Plays short tones for game events. Uses winsound on Windows and
    falls back to the cross-platform Tk system bell everywhere else."""

    def __init__(self, root):
        self.root = root
        self.enabled = True

    def toggle(self):
        self.enabled = not self.enabled
        return self.enabled

    def _beep(self, freq, dur):
        if not self.enabled:
            return
        try:
            if HAS_WINSOUND:
                winsound.Beep(freq, dur)
            else:
                self.root.bell()
        except Exception:
            pass

    def click(self):
        self._beep(523, 60)

    def win(self):
        self._beep(880, 250)

    def draw(self):
        self._beep(300, 250)

    def error(self):
        self._beep(150, 120)


# ---------------------------------------------------------------------------
# Themes
# ---------------------------------------------------------------------------
THEMES = {
    "light": {
        "bg": "#eef1f8",
        "fg": "#1b1f3b",
        "panel_bg": "#ffffff",
        "field_bg": "#f3f5fb",
        "accent": "#4361ee",
        "accent_fg": "#ffffff",
        "btn_active": "#dde3f5",
        "board_bg": "#dfe3ef",
        "x_color": "#e63946",
        "o_color": "#1d6fe0",
        "win_bg": "#52c97f",
        "muted": "#7d839c",
        "border": "#d7dbe8",
    },
    "dark": {
        "bg": "#0b0e1c",
        "fg": "#eef1f8",
        "panel_bg": "#161a31",
        "field_bg": "#101323",
        "accent": "#7c5cff",
        "accent_fg": "#ffffff",
        "btn_active": "#272c52",
        "board_bg": "#1c2042",
        "x_color": "#ff5d73",
        "o_color": "#2fd9ff",
        "win_bg": "#22c55e",
        "muted": "#7d84ab",
        "border": "#2a2e55",
    },
}

WINNING_COMBOS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6),
]


# ---------------------------------------------------------------------------
# Pure game-logic helpers (operate on a 9-item list of "", "X" or "O")
# ---------------------------------------------------------------------------
def get_winner(board):
    for combo in WINNING_COMBOS:
        a, b, c = combo
        if board[a] and board[a] == board[b] == board[c]:
            return board[a], combo
    return None, None


def is_draw(board):
    winner, _ = get_winner(board)
    return winner is None and "" not in board


def best_move_easy(board):
    empty = [i for i in range(9) if board[i] == ""]
    return random.choice(empty)


def best_move_medium(board, computer_mark, human_mark):
    empty = [i for i in range(9) if board[i] == ""]

    for i in empty:
        board[i] = computer_mark
        winner, _ = get_winner(board)
        board[i] = ""
        if winner == computer_mark:
            return i

    for i in empty:
        board[i] = human_mark
        winner, _ = get_winner(board)
        board[i] = ""
        if winner == human_mark:
            return i

    preferred = [4, 0, 2, 6, 8, 1, 3, 5, 7]
    preferred_available = [p for p in preferred if p in empty]
    if preferred_available and random.random() < 0.7:
        return preferred_available[0]
    return random.choice(empty)


def _minimax(board, current_mark, computer_mark, human_mark, depth):
    winner, _ = get_winner(board)
    if winner == computer_mark:
        return 10 - depth
    if winner == human_mark:
        return depth - 10
    if "" not in board:
        return 0

    scores = []
    for i in range(9):
        if board[i] == "":
            board[i] = current_mark
            next_mark = human_mark if current_mark == computer_mark else computer_mark
            scores.append(_minimax(board, next_mark, computer_mark, human_mark, depth + 1))
            board[i] = ""
    return max(scores) if current_mark == computer_mark else min(scores)


def best_move_hard(board, computer_mark, human_mark):
    """Unbeatable AI using the Minimax algorithm."""
    best_score = -math.inf
    best_moves = []
    for i in range(9):
        if board[i] == "":
            board[i] = computer_mark
            score = _minimax(board, human_mark, computer_mark, human_mark, 1)
            board[i] = ""
            if score > best_score:
                best_score = score
                best_moves = [i]
            elif score == best_score:
                best_moves.append(i)
    return random.choice(best_moves)


# ---------------------------------------------------------------------------
# Main application
# ---------------------------------------------------------------------------
class TicTacToeApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Tic Tac Toe — Arena")
        self.root.geometry("600x840")
        self.root.minsize(560, 760)

        self.sound = SoundManager(root)
        self.theme = "dark"
        self.colors = THEMES[self.theme]

        # ---- settings (chosen on the menu screen) ----
        self.mode = "pvc"            # "pvp" or "pvc"
        self.difficulty = "hard"     # "easy" / "medium" / "hard"
        self.best_of = 3             # 3 or 5
        self.timer_enabled = True
        self.timer_seconds_setting = 15

        self.name_x_var = tk.StringVar(value="Player 1")
        self.name_o_var = tk.StringVar(value="Computer")
        self._saved_o_name = "Player 2"

        # ---- runtime state ----
        self.board = [""] * 9
        self.cell_buttons = []
        self.current = "X"
        self.round_starter = "X"
        self.score_x = 0
        self.score_o = 0
        self.draws = 0
        self.round_num = 1
        self.input_locked = False
        self.match_over = False
        self.timer_id = None
        self.time_left = 0

        # ---- theming bookkeeping ----
        self.card_frames = []          # (card_widget,) recolored to panel_bg
        self.card_title_labels = []    # recolored to accent color

        # Both screens live inside a scrollable canvas so that nothing
        # (e.g. the Start Game button, or New Game / Reset Match) can ever
        # get clipped off the bottom of the window — regardless of window
        # size, OS font scaling, or how many option cards are visible.
        self.menu_outer, self.menu_canvas, self.menu_frame, self.menu_vscroll = self._make_scrollable(root)
        self.game_outer, self.game_canvas, self.game_frame, self.game_vscroll = self._make_scrollable(root)

        self.build_menu_screen()
        self.build_game_screen()
        self.apply_theme()
        self.show_menu()

    # ------------------------------------------------------------------
    # Scrollable screen helper
    # ------------------------------------------------------------------
    def _make_scrollable(self, parent):
        """Wrap content in a vertically-scrollable canvas.

        Without this, a Frame packed straight into a fixed-size root window
        simply gets clipped once its content is taller than the window —
        the missing widgets aren't broken, they're just out of view with no
        way to reach them. Wrapping each screen in its own canvas lets the
        user scroll to anything that doesn't fit.
        """
        outer = tk.Frame(parent)

        canvas = tk.Canvas(outer, highlightthickness=0, bd=0)
        # Explicit width + later theming (see apply_theme) makes this a
        # bold, colored bar instead of a thin OS-default sliver that's
        # easy to miss against a light background.
        vscroll = tk.Scrollbar(outer, orient="vertical", width=14, command=canvas.yview)
        canvas.configure(yscrollcommand=vscroll.set)
        canvas.pack(side="left", fill="both", expand=True)
        vscroll.pack(side="right", fill="y")

        inner = tk.Frame(canvas)
        inner_id = canvas.create_window((0, 0), window=inner, anchor="nw")

        def _sync_scrollregion(event=None):
            canvas.configure(scrollregion=canvas.bbox("all"))

        def _sync_inner_width(event):
            # Keep the inner frame's width matched to the visible canvas
            # width so fill="x" packing inside it behaves the same as a
            # normal directly-packed frame would.
            canvas.itemconfig(inner_id, width=event.width)

        inner.bind("<Configure>", _sync_scrollregion)
        canvas.bind("<Configure>", _sync_inner_width)

        def _on_mousewheel(event):
            if event.num == 4:
                canvas.yview_scroll(-1, "units")
            elif event.num == 5:
                canvas.yview_scroll(1, "units")
            else:
                canvas.yview_scroll(-1 if event.delta > 0 else 1, "units")

        def _bind_wheel(_event):
            canvas.bind_all("<MouseWheel>", _on_mousewheel)
            canvas.bind_all("<Button-4>", _on_mousewheel)
            canvas.bind_all("<Button-5>", _on_mousewheel)

        def _unbind_wheel(_event):
            canvas.unbind_all("<MouseWheel>")
            canvas.unbind_all("<Button-4>")
            canvas.unbind_all("<Button-5>")

        canvas.bind("<Enter>", _bind_wheel)
        canvas.bind("<Leave>", _unbind_wheel)

        return outer, canvas, inner, vscroll

    # ------------------------------------------------------------------
    # Screen switching
    # ------------------------------------------------------------------
    def show_menu(self):
        self.game_outer.pack_forget()
        self.menu_outer.pack(fill="both", expand=True)

    def show_game(self):
        self.menu_outer.pack_forget()
        self.game_outer.pack(fill="both", expand=True)

    # ------------------------------------------------------------------
    # Card helper (dashboard-style panel with an icon title)
    # ------------------------------------------------------------------
    def make_card(self, parent, icon, title_text):
        outer = tk.Frame(parent)
        outer.pack(fill="x", padx=24, pady=(0, 14))
        card = tk.Frame(outer, highlightthickness=1)
        card.pack(fill="x")
        inner = tk.Frame(card)
        inner.pack(fill="both", expand=True, padx=18, pady=14)
        title_lbl = tk.Label(inner, text=f"{icon}  {title_text}", font=("Arial", 12, "bold"))
        title_lbl.pack(anchor="w", pady=(0, 10))
        self.card_frames.append(card)
        self.card_title_labels.append(title_lbl)
        return outer, inner

    # ------------------------------------------------------------------
    # Menu screen
    # ------------------------------------------------------------------
    def build_menu_screen(self):
        f = self.menu_frame

        # ---- header / navbar ----
        top_bar = tk.Frame(f)
        top_bar.pack(fill="x", padx=20, pady=(16, 0))
        self.menu_brand_lbl = tk.Label(top_bar, text="⚡ GAME ARENA", font=("Arial", 10, "bold"))
        self.menu_brand_lbl.pack(side="left")
        self.menu_theme_btn = tk.Button(top_bar, text="🌙 Dark Mode", font=("Arial", 10),
                                         bd=0, padx=10, pady=4, command=self.toggle_theme)
        self.menu_theme_btn.pack(side="right", padx=4)
        self.menu_sound_btn = tk.Button(top_bar, text="🔊 Sound: On", font=("Arial", 10),
                                         bd=0, padx=10, pady=4, command=self.toggle_sound)
        self.menu_sound_btn.pack(side="right", padx=4)

        self.menu_divider = tk.Frame(f, height=2)
        self.menu_divider.pack(fill="x", padx=20, pady=(10, 0))

        title = tk.Label(f, text="TIC TAC TOE", font=("Arial", 30, "bold"))
        title.pack(pady=(18, 2))
        self.menu_underline = tk.Frame(f, width=160, height=4)
        self.menu_underline.pack(pady=(0, 8))
        subtitle = tk.Label(f, text="C O N F I G U R E   Y O U R   M A T C H", font=("Arial", 10, "bold"))
        subtitle.pack(pady=(0, 18))
        self.menu_title_lbl = title
        self.menu_subtitle_lbl = subtitle

        # ---- Game mode card ----
        _, mode_inner = self.make_card(f, "🎮", "Game Mode")
        mode_frame = tk.Frame(mode_inner)
        mode_frame.pack(fill="x")
        pvp_btn = tk.Button(mode_frame, text="🧑 vs 🧑\nPlayer vs Player", font=("Arial", 10, "bold"),
                             height=2, bd=0, command=lambda: self.select_mode("pvp"))
        pvc_btn = tk.Button(mode_frame, text="🧑 vs 🤖\nPlayer vs Computer", font=("Arial", 10, "bold"),
                             height=2, bd=0, command=lambda: self.select_mode("pvc"))
        pvp_btn.pack(side="left", expand=True, fill="x", padx=(0, 5))
        pvc_btn.pack(side="left", expand=True, fill="x", padx=(5, 0))
        self.mode_buttons = {"pvp": pvp_btn, "pvc": pvc_btn}

        # ---- Player names card ----
        _, names_inner = self.make_card(f, "🧑\u200d🤝\u200d🧑", "Player Names")
        names_frame = tk.Frame(names_inner)
        names_frame.pack(fill="x")
        x_lbl = tk.Label(names_frame, text="Player X:", font=("Arial", 10, "bold"))
        x_lbl.grid(row=0, column=0, sticky="w", pady=4)
        self.entry_x = tk.Entry(names_frame, textvariable=self.name_x_var, font=("Arial", 10),
                                 relief="flat", highlightthickness=1, bd=6)
        self.entry_x.grid(row=0, column=1, sticky="ew", pady=4, padx=(8, 0))
        o_lbl = tk.Label(names_frame, text="Player O:", font=("Arial", 10, "bold"))
        o_lbl.grid(row=1, column=0, sticky="w", pady=4)
        self.entry_o = tk.Entry(names_frame, textvariable=self.name_o_var, font=("Arial", 10),
                                 relief="flat", highlightthickness=1, bd=6)
        self.entry_o.grid(row=1, column=1, sticky="ew", pady=4, padx=(8, 0))
        names_frame.columnconfigure(1, weight=1)
        self.name_field_labels = [x_lbl, o_lbl]
        if self.mode == "pvc":
            self.entry_o.config(state="disabled")

        # ---- AI difficulty card (hidden entirely in Player vs Player mode) ----
        self.difficulty_outer, diff_inner = self.make_card(f, "🤖", "AI Difficulty")
        diff_frame = tk.Frame(diff_inner)
        diff_frame.pack(fill="x")
        easy_btn = tk.Button(diff_frame, text="🟢 Easy", font=("Arial", 10, "bold"), bd=0,
                              command=lambda: self.select_difficulty("easy"))
        med_btn = tk.Button(diff_frame, text="🟡 Medium", font=("Arial", 10, "bold"), bd=0,
                             command=lambda: self.select_difficulty("medium"))
        hard_btn = tk.Button(diff_frame, text="🔴 Hard (Minimax)", font=("Arial", 10, "bold"), bd=0,
                              command=lambda: self.select_difficulty("hard"))
        easy_btn.pack(side="left", expand=True, fill="x", padx=2)
        med_btn.pack(side="left", expand=True, fill="x", padx=2)
        hard_btn.pack(side="left", expand=True, fill="x", padx=2)
        self.diff_buttons = {"easy": easy_btn, "medium": med_btn, "hard": hard_btn}

        # ---- Match length card ----
        self.bestof_outer, bo_inner = self.make_card(f, "🏆", "Match Length")
        bo_frame = tk.Frame(bo_inner)
        bo_frame.pack(fill="x")
        bo3_btn = tk.Button(bo_frame, text="Best of 3", font=("Arial", 10, "bold"), bd=0,
                             command=lambda: self.select_bestof(3))
        bo5_btn = tk.Button(bo_frame, text="Best of 5", font=("Arial", 10, "bold"), bd=0,
                             command=lambda: self.select_bestof(5))
        bo3_btn.pack(side="left", expand=True, fill="x", padx=(0, 5))
        bo5_btn.pack(side="left", expand=True, fill="x", padx=(5, 0))
        self.bestof_buttons = {3: bo3_btn, 5: bo5_btn}

        # ---- Turn timer card ----
        _, timer_inner = self.make_card(f, "⏱", "Turn Timer")
        self.timer_toggle_btn = tk.Button(timer_inner, text="Timer: ON", font=("Arial", 10, "bold"),
                                           bd=0, command=self.toggle_timer_enabled)
        self.timer_toggle_btn.pack(fill="x", pady=(0, 8))
        timer_sec_row = tk.Frame(timer_inner)
        timer_sec_row.pack(fill="x")
        t10 = tk.Button(timer_sec_row, text="10s", font=("Arial", 10, "bold"), bd=0,
                         command=lambda: self.select_timer_seconds(10))
        t15 = tk.Button(timer_sec_row, text="15s", font=("Arial", 10, "bold"), bd=0,
                         command=lambda: self.select_timer_seconds(15))
        t20 = tk.Button(timer_sec_row, text="20s", font=("Arial", 10, "bold"), bd=0,
                         command=lambda: self.select_timer_seconds(20))
        t10.pack(side="left", expand=True, fill="x", padx=2)
        t15.pack(side="left", expand=True, fill="x", padx=2)
        t20.pack(side="left", expand=True, fill="x", padx=2)
        self.timer_sec_buttons = {10: t10, 15: t15, 20: t20}

        self.start_btn = tk.Button(f, text="▶  START GAME", font=("Arial", 15, "bold"), bd=0,
                                    height=2, command=self.start_match)
        self.start_btn.pack(fill="x", padx=24, pady=(4, 24))

        # Show/hide the AI difficulty card based on the initial mode
        self.update_difficulty_visibility()

    # ------------------------------------------------------------------
    # Game screen
    # ------------------------------------------------------------------
    def build_game_screen(self):
        f = self.game_frame

        top_bar = tk.Frame(f)
        top_bar.pack(fill="x", padx=20, pady=(16, 0))
        self.back_btn = tk.Button(top_bar, text="← Menu", font=("Arial", 10, "bold"), bd=0,
                                   padx=10, pady=4, command=self.back_to_menu)
        self.back_btn.pack(side="left")
        self.game_theme_btn = tk.Button(top_bar, text="🌙", font=("Arial", 10), bd=0, width=3,
                                         command=self.toggle_theme)
        self.game_theme_btn.pack(side="right", padx=2)
        self.game_sound_btn = tk.Button(top_bar, text="🔊", font=("Arial", 10), bd=0, width=3,
                                         command=self.toggle_sound)
        self.game_sound_btn.pack(side="right", padx=2)

        self.game_divider = tk.Frame(f, height=2)
        self.game_divider.pack(fill="x", padx=20, pady=(10, 14))

        # ---- VS scoreboard HUD ----
        hud = tk.Frame(f)
        hud.pack(fill="x", padx=20)
        hud.columnconfigure(0, weight=1)
        hud.columnconfigure(1, weight=0)
        hud.columnconfigure(2, weight=1)

        self.score_x_card = tk.Frame(hud, highlightthickness=2)
        self.score_x_card.grid(row=0, column=0, sticky="ew", padx=(0, 8))
        self.score_x_name_lbl = tk.Label(self.score_x_card, text="Player 1 (X)", font=("Arial", 11, "bold"))
        self.score_x_name_lbl.pack(pady=(10, 0))
        self.score_x_val_lbl = tk.Label(self.score_x_card, text="0", font=("Arial", 28, "bold"))
        self.score_x_val_lbl.pack(pady=(0, 10))

        self.vs_lbl = tk.Label(hud, text="VS", font=("Arial", 16, "bold"))
        self.vs_lbl.grid(row=0, column=1, padx=10)

        self.score_o_card = tk.Frame(hud, highlightthickness=2)
        self.score_o_card.grid(row=0, column=2, sticky="ew", padx=(8, 0))
        self.score_o_name_lbl = tk.Label(self.score_o_card, text="Player 2 (O)", font=("Arial", 11, "bold"))
        self.score_o_name_lbl.pack(pady=(10, 0))
        self.score_o_val_lbl = tk.Label(self.score_o_card, text="0", font=("Arial", 28, "bold"))
        self.score_o_val_lbl.pack(pady=(0, 10))

        self.draws_lbl = tk.Label(f, text="Draws: 0", font=("Arial", 10, "bold"))
        self.draws_lbl.pack(pady=(8, 0))

        self.round_badge = tk.Frame(f, highlightthickness=1)
        self.round_badge.pack(pady=(8, 6))
        self.round_lbl = tk.Label(self.round_badge, text="Round 1 • Best of 3 (first to 2 wins)",
                                   font=("Arial", 9, "bold"))
        self.round_lbl.pack(padx=14, pady=5)

        status_frame = tk.Frame(f)
        status_frame.pack(fill="x", padx=20, pady=(6, 6))
        self.status_lbl = tk.Label(status_frame, text="Player 1's Turn (X)", font=("Arial", 14, "bold"))
        self.status_lbl.pack(side="left")
        self.timer_lbl = tk.Label(status_frame, text="", font=("Arial", 14, "bold"))
        self.timer_lbl.pack(side="right")

        board_card = tk.Frame(f, highlightthickness=1)
        board_card.pack(pady=10)
        self.card_frames.append(board_card)
        self.board_frame = tk.Frame(board_card)
        self.board_frame.pack(padx=14, pady=14)
        for i in range(9):
            btn = tk.Button(self.board_frame, text="", font=("Arial", 32, "bold"), width=4, height=2,
                             bd=0, highlightthickness=2, command=lambda i=i: self.on_cell_click(i))
            btn.grid(row=i // 3, column=i % 3, padx=5, pady=5)
            btn.bind("<Enter>", lambda e, b=btn: self.on_cell_hover(b, True))
            btn.bind("<Leave>", lambda e, b=btn: self.on_cell_hover(b, False))
            self.cell_buttons.append(btn)

        bottom_frame = tk.Frame(f)
        bottom_frame.pack(fill="x", padx=20, pady=20)
        self.new_game_btn = tk.Button(bottom_frame, text="🏠 New Game", font=("Arial", 11, "bold"), bd=0,
                                       command=self.new_game)
        self.new_game_btn.pack(side="left", expand=True, fill="x", padx=(0, 5))
        self.reset_match_btn = tk.Button(bottom_frame, text="🔄 Reset Match", font=("Arial", 11, "bold"), bd=0,
                                          command=self.reset_match)
        self.reset_match_btn.pack(side="left", expand=True, fill="x", padx=(5, 0))

    # ------------------------------------------------------------------
    # Theming
    # ------------------------------------------------------------------
    def apply_theme(self):
        c = self.colors = THEMES[self.theme]
        self.root.config(bg=c["bg"])
        self.menu_canvas.config(bg=c["bg"])
        self.game_canvas.config(bg=c["bg"])
        self.menu_frame.config(bg=c["bg"])
        self.game_frame.config(bg=c["bg"])
        for vs in (self.menu_vscroll, self.game_vscroll):
            vs.config(bg=c["accent"], activebackground=c["accent"], troughcolor=c["panel_bg"],
                      highlightthickness=0, bd=0, elementborderwidth=0)

        # base pass: page-level backgrounds/text for everything
        self._recolor_widget(self.menu_frame, c)
        self._recolor_widget(self.game_frame, c)

        # card subtree pass: cards get the panel color instead of page color
        for card in self.card_frames:
            card.config(bg=c["panel_bg"], highlightbackground=c["border"], highlightcolor=c["border"])
            self._recolor_card_subtree(card, c)
        for lbl in self.card_title_labels:
            lbl.config(fg=c["accent"])

        # header accents
        self.menu_brand_lbl.config(bg=c["bg"], fg=c["accent"])
        self.menu_divider.config(bg=c["accent"])
        self.menu_underline.config(bg=c["accent"])
        self.menu_title_lbl.config(bg=c["bg"], fg=c["fg"])
        self.menu_subtitle_lbl.config(bg=c["bg"], fg=c["muted"])
        self.game_divider.config(bg=c["accent"])
        for lbl in self.name_field_labels:
            lbl.config(bg=c["panel_bg"], fg=c["fg"])
        self.entry_x.config(bg=c["field_bg"], fg=c["fg"], insertbackground=c["fg"],
                             highlightbackground=c["border"], highlightcolor=c["accent"])
        self.entry_o.config(bg=c["field_bg"], fg=c["fg"], insertbackground=c["fg"],
                             highlightbackground=c["border"], highlightcolor=c["accent"],
                             disabledbackground=c["panel_bg"], disabledforeground=c["muted"])

        # primary buttons
        self.start_btn.config(bg=c["accent"], fg=c["accent_fg"], activebackground=c["accent"])
        self.back_btn.config(bg=c["bg"], fg=c["fg"], activebackground=c["btn_active"])
        for b in (self.menu_theme_btn, self.menu_sound_btn, self.game_theme_btn, self.game_sound_btn):
            b.config(bg=c["bg"], fg=c["fg"], activebackground=c["btn_active"])
        for b in (self.new_game_btn, self.reset_match_btn):
            b.config(bg=c["panel_bg"], fg=c["fg"], highlightthickness=1,
                     highlightbackground=c["border"], activebackground=c["btn_active"])

        # status / timer / round badge
        self.status_lbl.config(bg=c["bg"], fg=c["fg"])
        self.timer_lbl.config(bg=c["bg"], fg=c["accent"])
        self.draws_lbl.config(bg=c["bg"], fg=c["muted"])
        self.round_badge.config(bg=c["panel_bg"], highlightbackground=c["border"])
        self.round_lbl.config(bg=c["panel_bg"], fg=c["muted"])

        # VS scoreboard HUD
        self.vs_lbl.config(bg=c["bg"], fg=c["muted"])
        self.score_x_card.config(bg=c["panel_bg"], highlightbackground=c["x_color"])
        self.score_x_name_lbl.config(bg=c["panel_bg"], fg=c["fg"])
        self.score_x_val_lbl.config(bg=c["panel_bg"], fg=c["x_color"])
        self.score_o_card.config(bg=c["panel_bg"], highlightbackground=c["o_color"])
        self.score_o_name_lbl.config(bg=c["panel_bg"], fg=c["fg"])
        self.score_o_val_lbl.config(bg=c["panel_bg"], fg=c["o_color"])

        self.refresh_choice_styles()
        self.style_cells()
        self.update_toggle_button_texts()

    def _recolor_widget(self, widget, c):
        for child in widget.winfo_children():
            cls = child.winfo_class()
            if cls == "Frame":
                child.config(bg=c["bg"])
            elif cls == "Label":
                child.config(bg=c["bg"], fg=c["fg"])
            elif cls == "Button":
                child.config(bg=c["panel_bg"], fg=c["fg"],
                              activebackground=c["btn_active"], activeforeground=c["fg"],
                              disabledforeground=c["muted"])
            elif cls == "Entry":
                child.config(bg=c["panel_bg"], fg=c["fg"], insertbackground=c["fg"],
                              disabledbackground=c["bg"], disabledforeground=c["muted"])
            self._recolor_widget(child, c)

    def _recolor_card_subtree(self, widget, c):
        for child in widget.winfo_children():
            cls = child.winfo_class()
            if cls == "Frame":
                child.config(bg=c["panel_bg"])
            elif cls == "Label":
                child.config(bg=c["panel_bg"], fg=c["fg"])
            elif cls == "Button":
                child.config(bg=c["panel_bg"], disabledforeground=c["muted"])
            self._recolor_card_subtree(child, c)

    def refresh_choice_styles(self):
        c = self.colors

        def style(buttons_dict, selected_key, enabled=True):
            for k, btn in buttons_dict.items():
                state = "normal" if enabled else "disabled"
                if enabled and k == selected_key:
                    btn.config(bg=c["accent"], fg=c["accent_fg"], state=state,
                               highlightthickness=2, highlightbackground=c["accent"],
                               disabledforeground=c["muted"])
                elif enabled:
                    btn.config(bg=c["panel_bg"], fg=c["fg"], state=state,
                               highlightthickness=1, highlightbackground=c["border"],
                               disabledforeground=c["muted"])
                else:
                    btn.config(bg=c["panel_bg"], fg=c["muted"], state=state,
                               highlightthickness=1, highlightbackground=c["border"],
                               disabledforeground=c["muted"])

        style(self.mode_buttons, self.mode)
        style(self.diff_buttons, self.difficulty, enabled=(self.mode == "pvc"))
        style(self.bestof_buttons, self.best_of)
        style(self.timer_sec_buttons, self.timer_seconds_setting, enabled=self.timer_enabled)
        self.timer_toggle_btn.config(
            bg=c["accent"] if self.timer_enabled else c["panel_bg"],
            fg=c["accent_fg"] if self.timer_enabled else c["fg"],
            highlightthickness=2 if self.timer_enabled else 1,
            highlightbackground=c["accent"] if self.timer_enabled else c["border"],
        )

    def style_cells(self):
        c = self.colors
        for i, btn in enumerate(self.cell_buttons):
            mark = self.board[i]
            if mark == "":
                btn.config(bg=c["board_bg"], fg=c["fg"], text="",
                           highlightbackground=c["border"], highlightthickness=2)
            elif mark == "X":
                btn.config(bg=c["board_bg"], fg=c["x_color"], text="X",
                           highlightbackground=c["x_color"], highlightthickness=2)
            else:
                btn.config(bg=c["board_bg"], fg=c["o_color"], text="O",
                           highlightbackground=c["o_color"], highlightthickness=2)

    def update_toggle_button_texts(self):
        theme_txt_long = "☀️ Light Mode" if self.theme == "dark" else "🌙 Dark Mode"
        theme_txt_short = "☀️" if self.theme == "dark" else "🌙"
        self.menu_theme_btn.config(text=theme_txt_long)
        self.game_theme_btn.config(text=theme_txt_short)
        sound_txt_long = "🔊 Sound: On" if self.sound.enabled else "🔈 Sound: Off"
        sound_txt_short = "🔊" if self.sound.enabled else "🔈"
        self.menu_sound_btn.config(text=sound_txt_long)
        self.game_sound_btn.config(text=sound_txt_short)
        self.timer_toggle_btn.config(text="Timer: ON" if self.timer_enabled else "Timer: OFF")

    def toggle_theme(self):
        self.theme = "dark" if self.theme == "light" else "light"
        self.apply_theme()

    def toggle_sound(self):
        self.sound.toggle()
        self.update_toggle_button_texts()

    # ------------------------------------------------------------------
    # Menu interactions
    # ------------------------------------------------------------------
    def update_difficulty_visibility(self):
        """Show the AI Difficulty card only in Player vs Computer mode."""
        if self.mode == "pvc":
            if not self.difficulty_outer.winfo_ismapped():
                self.difficulty_outer.pack(fill="x", padx=24, pady=(0, 14), before=self.bestof_outer)
        else:
            if self.difficulty_outer.winfo_ismapped():
                self.difficulty_outer.pack_forget()

    def select_mode(self, mode):
        if self.mode == mode:
            return
        self.mode = mode
        self.sound.click()
        if mode == "pvc":
            current_o = self.name_o_var.get().strip()
            if current_o and current_o != "Computer":
                self._saved_o_name = current_o
            self.name_o_var.set("Computer")
            self.entry_o.config(state="disabled")
        else:
            self.name_o_var.set(self._saved_o_name or "Player 2")
            self.entry_o.config(state="normal")
        self.update_difficulty_visibility()
        self.refresh_choice_styles()

    def select_difficulty(self, level):
        self.difficulty = level
        self.sound.click()
        self.refresh_choice_styles()

    def select_bestof(self, n):
        self.best_of = n
        self.sound.click()
        self.refresh_choice_styles()

    def toggle_timer_enabled(self):
        self.timer_enabled = not self.timer_enabled
        self.sound.click()
        self.refresh_choice_styles()
        self.update_toggle_button_texts()

    def select_timer_seconds(self, n):
        self.timer_seconds_setting = n
        self.sound.click()
        self.refresh_choice_styles()

    # ------------------------------------------------------------------
    # Match / round flow
    # ------------------------------------------------------------------
    def wins_needed(self):
        return self.best_of // 2 + 1

    def start_match(self):
        name_x = self.name_x_var.get().strip() or "Player 1"
        name_o = "Computer" if self.mode == "pvc" else (self.name_o_var.get().strip() or "Player 2")
        self.name_x_var.set(name_x)
        self.name_o_var.set(name_o)

        self.score_x = 0
        self.score_o = 0
        self.draws = 0
        self.round_num = 1
        self.round_starter = "X"
        self.match_over = False
        self.sound.click()
        self.update_scoreboard()
        self.show_game()
        self.start_round()

    def start_round(self):
        self.cancel_timer()
        self.board = [""] * 9
        self.input_locked = False
        self.current = self.round_starter

        for btn in self.cell_buttons:
            btn.config(text="", state="normal")
        self.style_cells()
        self.update_round_label()
        self.update_status_for_turn()

        if self.mode == "pvc" and self.current == "O":
            self.input_locked = True
            self.timer_lbl.config(text="")
            self.root.after(500, self.computer_turn)
        else:
            self.start_timer()

    def update_status_for_turn(self):
        name = self.name_x_var.get() if self.current == "X" else self.name_o_var.get()
        self.status_lbl.config(text=f"{name}'s Turn ({self.current})")

    def update_round_label(self):
        self.round_lbl.config(
            text=f"Round {self.round_num} • Best of {self.best_of} (first to {self.wins_needed()} wins)"
        )

    def update_scoreboard(self):
        self.score_x_name_lbl.config(text=f"{self.name_x_var.get()} (X)")
        self.score_x_val_lbl.config(text=str(self.score_x))
        self.score_o_name_lbl.config(text=f"{self.name_o_var.get()} (O)")
        self.score_o_val_lbl.config(text=str(self.score_o))
        self.draws_lbl.config(text=f"Draws: {self.draws}")
        self.update_round_label()

    # ------------------------------------------------------------------
    # Moves
    # ------------------------------------------------------------------
    def on_cell_hover(self, btn, entering):
        idx = self.cell_buttons.index(btn)
        if self.board[idx] != "" or self.input_locked:
            return
        c = self.colors
        if entering:
            btn.config(bg=c["btn_active"], highlightbackground=c["accent"])
        else:
            btn.config(bg=c["board_bg"], highlightbackground=c["border"])

    def on_cell_click(self, i):
        if self.input_locked or self.board[i] != "":
            return
        self.place_mark(i, self.current)

    def place_mark(self, index, mark):
        self.cancel_timer()
        self.board[index] = mark
        c = self.colors
        self.cell_buttons[index].config(text=mark, state="disabled",
                                         fg=c["x_color"] if mark == "X" else c["o_color"],
                                         highlightbackground=c["x_color"] if mark == "X" else c["o_color"])
        self.sound.click()
        self.after_move_checks(mark)

    def after_move_checks(self, mark):
        winner, combo = get_winner(self.board)
        if winner:
            self.handle_round_end("win", winner, combo)
            return
        if is_draw(self.board):
            self.handle_round_end("draw")
            return

        self.current = "O" if mark == "X" else "X"
        self.input_locked = False
        self.update_status_for_turn()

        if self.mode == "pvc" and self.current == "O":
            self.input_locked = True
            self.timer_lbl.config(text="")
            self.root.after(450, self.computer_turn)
        else:
            self.start_timer()

    def computer_turn(self):
        if self.difficulty == "easy":
            move = best_move_easy(self.board)
        elif self.difficulty == "medium":
            move = best_move_medium(self.board, "O", "X")
        else:
            move = best_move_hard(self.board, "O", "X")
        self.place_mark(move, "O")

    # ------------------------------------------------------------------
    # Turn timer
    # ------------------------------------------------------------------
    def start_timer(self):
        if not self.timer_enabled:
            self.timer_lbl.config(text="")
            return
        self.time_left = self.timer_seconds_setting
        self.timer_lbl.config(text=f"⏱ {self.time_left}s")
        self.timer_id = self.root.after(1000, self.timer_tick)

    def timer_tick(self):
        self.time_left -= 1
        if self.time_left <= 0:
            self.timer_lbl.config(text="⏱ 0s")
            self.timer_id = None
            self.handle_timeout()
            return
        self.timer_lbl.config(text=f"⏱ {self.time_left}s")
        self.timer_id = self.root.after(1000, self.timer_tick)

    def cancel_timer(self):
        if self.timer_id is not None:
            try:
                self.root.after_cancel(self.timer_id)
            except Exception:
                pass
            self.timer_id = None
        self.timer_lbl.config(text="")

    def handle_timeout(self):
        self.sound.error()
        empties = [i for i in range(9) if self.board[i] == ""]
        if not empties:
            return
        move = random.choice(empties)
        mark = self.current
        self.status_lbl.config(text=f"Time's up! A random move was made for {mark}.")
        self.place_mark(move, mark)

    # ------------------------------------------------------------------
    # Round / match completion
    # ------------------------------------------------------------------
    def handle_round_end(self, result, winner=None, combo=None):
        self.cancel_timer()
        self.input_locked = True
        for btn in self.cell_buttons:
            btn.config(state="disabled")

        c = self.colors
        if result == "win":
            for idx in combo:
                self.cell_buttons[idx].config(bg=c["win_bg"], highlightbackground=c["win_bg"])
            if winner == "X":
                self.score_x += 1
                winner_name = self.name_x_var.get()
            else:
                self.score_o += 1
                winner_name = self.name_o_var.get()
            self.status_lbl.config(text=f"{winner_name} wins this round!")
            self.sound.win()
        else:
            self.draws += 1
            self.status_lbl.config(text="It's a draw!")
            self.sound.draw()

        self.update_scoreboard()
        self.root.update_idletasks()
        self.root.after(300, lambda: self.show_round_result_dialog(result, winner))

    def show_round_result_dialog(self, result, winner):
        wn = self.wins_needed()
        if result == "win":
            name = self.name_x_var.get() if winner == "X" else self.name_o_var.get()
            msg = f"{name} wins Round {self.round_num}!"
        else:
            msg = f"Round {self.round_num} is a draw!"

        if self.score_x >= wn or self.score_o >= wn:
            messagebox.showinfo("Round Result", msg)
            champ = self.name_x_var.get() if self.score_x > self.score_o else self.name_o_var.get()
            self.match_over = True
            messagebox.showinfo(
                "Match Over",
                f"🏆 {champ} wins the match!\n\n"
                f"Final score — {self.name_x_var.get()}: {self.score_x}  |  "
                f"{self.name_o_var.get()}: {self.score_o}  |  Draws: {self.draws}",
            )
            self.status_lbl.config(text=f"{champ} wins the match! 🏆")
        else:
            messagebox.showinfo("Round Result", msg)
            self.round_num += 1
            self.round_starter = "O" if self.round_starter == "X" else "X"
            self.start_round()

    # ------------------------------------------------------------------
    # Bottom controls
    # ------------------------------------------------------------------
    def new_game(self):
        """Returns to the menu so a new game can be configured."""
        self.sound.click()
        self.cancel_timer()
        self.show_menu()

    def reset_match(self):
        if not messagebox.askyesno("Reset Match", "Reset the scoreboard and start a new match?"):
            return
        self.sound.click()
        self.score_x = 0
        self.score_o = 0
        self.draws = 0
        self.round_num = 1
        self.round_starter = "X"
        self.match_over = False
        self.update_scoreboard()
        self.start_round()

    def back_to_menu(self):
        self.cancel_timer()
        self.show_menu()


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    root = tk.Tk()
    app = TicTacToeApp(root)
    root.mainloop()